# NBA Player Efficiency Analysis — Exploration Notebook

This notebook walks through the data collection and cleaning pipeline,
verifies the API connection, and produces sample visualizations.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collection import (
    find_player, get_shot_chart, get_career_stats,
    season_range, collect_player_data,
)
from src.data_cleaning import (
    clean_shot_chart, aggregate_by_zone, aggregate_by_grid,
    export_for_tableau,
)

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Verify API Connection

In [ ]:
# Look up a well-known player to verify the API is reachable
player = find_player('LeBron James')
print(f"Found: {player['full_name']} (ID: {player['id']})")

## 2. Pull Sample Shot Chart

In [ ]:
shots = get_shot_chart(player['id'], season='2023-24')
print(f"Rows: {len(shots)}")
shots.head()

## 3. Clean & Enrich

In [ ]:
cleaned = clean_shot_chart(shots)
cleaned[['PLAYER_NAME', 'LOC_X', 'LOC_Y', 'SHOT_MADE_FLAG', 'RESULT',
         'SHOT_ZONE_BASIC', 'CALC_DISTANCE_FT']].head(10)

## 4. Basic Shot Chart Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
colors = cleaned['SHOT_MADE_FLAG'].map({1: 'green', 0: 'red'})
ax.scatter(cleaned['LOC_X'], cleaned['LOC_Y'], c=colors, alpha=0.4, s=8)
ax.set_xlim(-250, 250)
ax.set_ylim(-50, 470)
ax.set_aspect('equal')
ax.set_title(f"{player['full_name']} — 2023-24 Shot Chart")
ax.set_xlabel('LOC_X (tenths of ft)')
ax.set_ylabel('LOC_Y (tenths of ft)')
plt.tight_layout()
plt.show()

## 5. Zone Aggregation

In [ ]:
zones = aggregate_by_zone(cleaned)
zones.sort_values('FGA', ascending=False)

## 6. Grid Heatmap

In [ ]:
grid = aggregate_by_grid(cleaned, bin_size=25)

fig, ax = plt.subplots(figsize=(8, 7))
scatter = ax.scatter(
    grid['grid_x'], grid['grid_y'],
    c=grid['FG_PCT'], cmap='RdYlGn',
    s=grid['FGA'] * 2, alpha=0.7,
    edgecolors='gray', linewidth=0.3,
)
plt.colorbar(scatter, ax=ax, label='FG%')
ax.set_xlim(-250, 250)
ax.set_ylim(-50, 470)
ax.set_aspect('equal')
ax.set_title(f"{player['full_name']} — Shooting Efficiency Heatmap")
plt.tight_layout()
plt.show()

## 7. Export for Tableau

Run this cell to save cleaned shot data and grid aggregation to `data/exports/`.

In [ ]:
export_for_tableau(cleaned, 'lebron_2023_24_shots.csv')
export_for_tableau(grid, 'lebron_2023_24_grid.csv')
print('Exported to data/exports/')